In [ ]:
# !pip install optimum[gptq]
!pip install auto_gptq
!pip install optimum
!pip install transformers
!pip install huggingface_hub
!pip install -U accelerate bitsandbytes datasets peft transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 16.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 28.0 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.7/422.7 kB 6.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
nltk.download('punkt')

# Initialize model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/CodeQwen1.5-7B-Chat",
    torch_dtype="auto",
    device_map="auto"
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("Qwen/CodeQwen1.5-7B-Chat")

def translate_verilog_to_c(verilog_code):
    prompt = f"""Translate the following Verilog code to C code. 
    Only output the C code, without any explanations or additional text.
    
    Verilog code:
    {verilog_code}"""
    
    messages = [
        {"role": "system", "content": "You are a code translation assistant that converts Verilog to C code."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=1024,
            temperature=0.7,
            do_sample=True
        )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response.strip()

def load_data(src_file, tgt_file):
    with open(src_file, 'r') as sf, open(tgt_file, 'r') as tf:
        src_lines = sf.readlines()
        tgt_lines = tf.readlines()
    assert len(src_lines) == len(tgt_lines), "Mismatch in number of source and target lines"
    return [(src.strip(), tgt.strip()) for src, tgt in zip(src_lines, tgt_lines)]

# Load test data
test_data = load_data('/kaggle/input/code-verilog-to-c/test.src', '/kaggle/input/code-verilog-to-c/test.tgt')

# Generate predictions and calculate BLEU score
predictions = []
references = []

for verilog_code, target_c_code in tqdm(test_data, desc="Generating predictions"):
    c_code = translate_verilog_to_c(verilog_code)
    predictions.append(c_code.split())
    references.append([target_c_code.split()])

# Calculate BLEU score
smoothie = SmoothingFunction().method4
bleu_score = corpus_bleu(references, predictions, smoothing_function=smoothie)
print(f"BLEU Score: {bleu_score}")

# Print some example translations
num_examples = 5
print("\nExample Translations:")
for i in range(min(num_examples, len(test_data))):
    print(f"\nExample {i + 1}:")
    print(f"Source (Verilog):\n{test_data[i][0]}")
    print(f"\nTarget (C):\n{test_data[i][1]}")
    print(f"\nPrediction (C):\n{' '.join(predictions[i])}")
    print("-" * 50)

# Save predictions to a file
with open('predictions.txt', 'w') as f:
    for pred in predictions:
        f.write(' '.join(pred) + '\n')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/31.7k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.89G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/4.46M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

Generating predictions: 100%|██████████| 5/5 [03:43<00:00, 44.64s/it]

BLEU Score: 0.012685502373194204

Example Translations:

Example 1:
Source (Verilog):
module entry_point ( , ap_start, , ap_done, , ap_idle, , ap_ready, , speed, , time_r, , temptemp1, , temptemp1_ap_vld, , ap_return , ); , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] speed; , input [31:0] time_r; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] ap_return; , reg temptemp1_ap_vld; , wire [31:0] distance_fu_45_p2; , wire ap_ce_reg; , entry_point_mul_32s_32s_32_1_1 #( , .ID( 1 ), , .NUM_STAGE( 1 ), , .din0_WIDTH( 32 ), , .din1_WIDTH( 32 ), , .dout_WIDTH( 32 )) , mul_32s_32s_32_1_1_U1( , .din0(time_r), , .din1(speed), , .dout(distance_fu_45_p2) , ); , always @ (*) begin , if ((ap_start == 1'b1)) begin , temptemp1_ap_vld = 1'b1; , end else begin , temptemp1_ap_vld = 1'b0; , end , end , assign ap_done = ap_start; , assign ap_idle = 1'b1; , assign ap_ready = ap_start; , assign ap_return = 32'd0; , assign temptemp1 = distance_fu_45_